In [114]:
import pandas as pd
import numpy as np

In [115]:
#sve trke od 2018
trke = pd.read_csv('tables\\all_races.csv')

In [116]:
trke.columns

Index(['season', 'round', 'race_name', 'date', 'time', 'circuit', 'country'], dtype='object')

In [117]:
#izbacivanje nepotrebnih kolona
trke = trke.drop(['date', 'time', 'country'], axis=1)

In [118]:
# trka season 2024 round 3 ima 19 klasifikovanih vozaca, izbaciti je 
trke = trke[~((trke['season'] == 2024) & (trke['round'] == 3))]

In [119]:
#na kojim se stazama odrzavaju trke
staze = set(zip(trke['circuit']))

In [120]:
staze = pd.DataFrame(staze)

In [121]:
#cuvamo samo imena staza
staze.to_csv('tables\\non_processed_circuits.csv', index=False)
print(f"Sacuvano")

Sacuvano


In [122]:
#pre ovoga rucno unete vrednosti: duzina kruga, broj krugova, broj krivina
staze = pd.read_csv('tables\\processed_circuits.csv')

In [123]:
#spjaju se podaci o stazama i trkama
trke = trke.merge(staze, on='circuit', how='left')    

In [124]:
#ucitavaju se podaci o vremenskim uslovima 
vreme = pd.read_csv('tables\\all_weather.csv')
vreme = vreme.drop(['date', 'time', 'race_name'], axis=1)

In [125]:
#primenjujemo binning za kisu
bins = [ -1, 0, 1, 5, 100 ]
labels = [0, 1, 2, 3]
vreme['rain_bin'] = pd.cut(vreme['Rainfall'], bins=bins, labels=labels)


In [126]:
#binning za jedan sample vetra
def wind_dir_bin(deg):
    if (deg >= 337.5) or (deg < 22.5):
        return 0  # N
    elif deg < 67.5:
        return 1  # NE
    elif deg < 112.5:
        return 2  # E
    elif deg < 157.5:
        return 3  # SE
    elif deg < 202.5:
        return 4  # S
    elif deg < 247.5:
        return 5  # SW
    elif deg < 292.5:
        return 6  # W
    else:
        return 7  # NW

In [127]:
#primenjujemo binning na sve vrednosti u koloni 'WindDirection'
vreme['wind_direction'] = vreme['WindDirection'].apply(wind_dir_bin)

In [128]:
#izbacujemo stare, nepotebne kolone
vreme = vreme.drop(['Rainfall', 'WindDirection'], axis=1)

In [129]:
#dodajemo podatke o vremenu na staze i trke
trke = trke.merge(vreme, on=['season', 'round'], how='left')    

In [130]:
#inicijalizovanje vrednosti da li je vikend sprint
trke['is_sprint_weekend'] = False

In [131]:
#u formatu [season, round]
sprint_vikendi = [
    [2021, 10], 
    [2021, 14], 
    [2021, 19], 

    [2022, 4], 
    [2022, 11], 
    [2022, 21], 

    [2023, 4], 
    [2023, 10], 
    [2023, 13], 
    [2023, 18], 
    [2023, 19], 
    [2023, 21], 

    [2024, 5],
    [2024, 6], 
    [2024, 11], 
    [2024, 19], 
    [2024, 21], 
    [2024, 23] 
]

In [132]:
#kada je sprint vikend promeniti iz fales u true
for i in range(len(trke)):
    if [trke.loc[i]['season'], trke.loc[i]['round']] in sprint_vikendi:
        trke.at[i, 'is_sprint_weekend'] = True

In [133]:
trke.head()

,season,round,race_name,circuit,lap_length,number_of_laps,number_of_corners,pit_lane_time_loss,average_overtakes_per_circuit,average_pit_stops_per_race,AirTemp,Humidity,Pressure,TrackTemp,WindSpeed,rain_bin,wind_direction,is_sprint_weekend
0,2018,1,Australian Grand Prix,Albert Park Grand Prix Circuit,5.303,58,14,NaN,NaN,NaN,24.077477,30.915315,997.003604,36.324324,3.691892,1,7,False
1,2018,2,Bahrain Grand Prix,Bahrain International Circuit,5.412,57,15,NaN,NaN,NaN,27.982524,47.363107,1009.494175,32.198058,0.958252,0,4,False
2,2018,3,Chinese Grand Prix,Shanghai International Circuit,5.451,56,16,NaN,NaN,NaN,19.446429,24.089286,1018.131250,37.019643,1.837500,1,3,False
3,2018,4,Azerbaijan Grand Prix,Baku City Circuit,6.003,51,20,,NaN,NaN,16.661404,45.651754,1021.913158,25.251754,2.222807,0,3,False
4,2018,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,4.655,66,14,NaN,NaN,NaN,16.050476,52.286667,1001.541905,32.339048,1.952381,1,2,False


In [134]:
#ucitavajne rezultata kvalifikacija odbacivanje nepotrebnih kolona
kvalifikacije = pd.read_csv('tables\\all_qualies.csv')
kvalifikacije = kvalifikacije.drop(['race_name','Q1_time', 'Q2_time', 
                                    'Q3_time'], axis=1)

In [135]:
#ucitavajne rezultata trka 
rezultati = pd.read_csv('tables\\all_race_results.csv')

In [136]:
#ucitavajne informacija o vozacima
vozaci = pd.read_csv('tables\\all_drivers.csv')

In [137]:
vozaci.columns

Index(['driverId', 'permanentNumber', 'code', 'url', 'givenName', 'familyName',
       'dateOfBirth', 'nationality'],
      dtype='object')

In [138]:
#sitne izmene da bi se nazivi kolona poklapali
vozaci = vozaci.rename(columns={'familyName': 'driver'})
vozaci = vozaci.drop(['code', 'url', 'givenName','dateOfBirth', 
                      'nationality'], axis=1)

In [139]:
rezultati.columns

Index(['season', 'round', 'raceName', 'date', 'circuit', 'position_race',
       'status', 'driver', 'constructor', 'grid', 'laps', 'total_time',
       'total_time_ms', 'points', 'fastest_lap_rank', 'fastest_lap_number',
       'fastest_lap_time', 'fastest_lap_speed', 'fastest_lap_speed_unit'],
      dtype='object')

In [140]:
#izbacivanje nepotrebnih kolona
rezultati = rezultati.drop(['raceName', 'date', 'circuit', 'total_time',
            'fastest_lap_speed_unit', 'fastest_lap_rank', 
            'fastest_lap_number', 'fastest_lap_time',
            'fastest_lap_speed'], axis=1)

In [141]:
vozaci.columns

Index(['driverId', 'permanentNumber', 'driver'], dtype='object')

In [142]:
#nedostajalo u tabeli sa vozačima
vozaci.loc[-1] = ['brendon_hartley', 28, 'Hartley']

In [143]:
#da budu svi podaci istog tipa
vozaci['permanentNumber'] = vozaci['permanentNumber'].astype(int)

In [144]:
#dodajemo vozače na rezultate trke, imacemo 20 puta vise redva
rezultati = pd.merge(rezultati, vozaci, on=['driver'], how='left')

In [145]:
#sitne izmene da bi se nazivi kolona poklapali
rezultati = rezultati.rename(columns={'permanentNumber': 'drivers_num'})

In [146]:
rezultati.tail()

,season,round,position_race,status,driver,constructor,grid,laps,total_time_ms,points,driverId,drivers_num
2974,2024,24,16,+1 Lap,Magnussen,Haas F1 Team,14,57,NaN,0.0,kevin_magnussen,20
2975,2024,24,17,Engine,Lawson,RB F1 Team,12,55,NaN,0.0,lawson,30
2976,2024,24,18,Collision damage,Bottas,Sauber,9,30,NaN,0.0,bottas,77
2977,2024,24,19,Engine,Colapinto,Williams,20,26,NaN,0.0,colapinto,43
2978,2024,24,20,Collision,Pérez,Red Bull,10,0,NaN,0.0,perez,11


In [147]:
#da budu svi podaci istog tipa
rezultati['season'] = rezultati['season'].astype(int)
rezultati['round'] = rezultati['round'].astype(int)
kvalifikacije['season'] = kvalifikacije['season'].astype(int)
kvalifikacije['round'] = kvalifikacije['round'].astype(int)
kvalifikacije['drivers_num'] = kvalifikacije['drivers_num'].astype(int)

In [148]:
#spajanje rezultata kvalifikacija i trka
rezultati = pd.merge(kvalifikacije, rezultati, on=['season', 'round', 'drivers_num'], how='outer', indicator=True)

In [149]:
rezultati.columns

Index(['season', 'round', 'drivers_num', 'position_quali', 'position_race',
       'status', 'driver', 'constructor', 'grid', 'laps', 'total_time_ms',
       'points', 'driverId', '_merge'],
      dtype='object')

In [150]:
rezultati = rezultati.rename(columns={'position_x': 'position_race'})
rezultati = rezultati.rename(columns={'position_y': 'position_quali'})

In [151]:
trke.head()

,season,round,race_name,circuit,lap_length,number_of_laps,number_of_corners,pit_lane_time_loss,average_overtakes_per_circuit,average_pit_stops_per_race,AirTemp,Humidity,Pressure,TrackTemp,WindSpeed,rain_bin,wind_direction,is_sprint_weekend
0,2018,1,Australian Grand Prix,Albert Park Grand Prix Circuit,5.303,58,14,NaN,NaN,NaN,24.077477,30.915315,997.003604,36.324324,3.691892,1,7,False
1,2018,2,Bahrain Grand Prix,Bahrain International Circuit,5.412,57,15,NaN,NaN,NaN,27.982524,47.363107,1009.494175,32.198058,0.958252,0,4,False
2,2018,3,Chinese Grand Prix,Shanghai International Circuit,5.451,56,16,NaN,NaN,NaN,19.446429,24.089286,1018.131250,37.019643,1.837500,1,3,False
3,2018,4,Azerbaijan Grand Prix,Baku City Circuit,6.003,51,20,,NaN,NaN,16.661404,45.651754,1021.913158,25.251754,2.222807,0,3,False
4,2018,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,4.655,66,14,NaN,NaN,NaN,16.050476,52.286667,1001.541905,32.339048,1.952381,1,2,False


In [152]:
#da budu svi podaci istog tipa
rezultati['season'] = rezultati['season'].astype(int)
rezultati['round'] = rezultati['round'].astype(int)
trke['season'] = trke['season'].astype(int)
trke['round'] = trke['round'].astype(int)

In [153]:
#spajanje zajednickih informacija za svaku trku i pojedniacnih informacija o vozacima
trke = pd.merge(rezultati, trke, on=['season', 'round'], how='left')

In [154]:
trke.columns

Index(['season', 'round', 'drivers_num', 'position_quali', 'position_race',
       'status', 'driver', 'constructor', 'grid', 'laps', 'total_time_ms',
       'points', 'driverId', '_merge', 'race_name', 'circuit', ' lap_length',
       ' number_of_laps', ' number_of_corners', ' pit_lane_time_loss',
       ' average_overtakes_per_circuit', ' average_pit_stops_per_race',
       'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindSpeed', 'rain_bin',
       'wind_direction', 'is_sprint_weekend'],
      dtype='object')

In [155]:
trke.head()

,season,round,drivers_num,position_quali,position_race,status,driver,constructor,grid,laps,...,average_overtakes_per_circuit,average_pit_stops_per_race,AirTemp,Humidity,Pressure,TrackTemp,WindSpeed,rain_bin,wind_direction,is_sprint_weekend
0,2018,1,2,12,9,Finished,Vandoorne,McLaren,11,58,...,NaN,NaN,24.077477,30.915315,997.003604,36.324324,3.691892,1,7.0,False
1,2018,1,3,5,4,Finished,Ricciardo,Red Bull,8,58,...,NaN,NaN,24.077477,30.915315,997.003604,36.324324,3.691892,1,7.0,False
2,2018,1,5,3,1,Finished,Vettel,Ferrari,3,58,...,NaN,NaN,24.077477,30.915315,997.003604,36.324324,3.691892,1,7.0,False
3,2018,1,7,2,3,Finished,Räikkönen,Ferrari,2,58,...,NaN,NaN,24.077477,30.915315,997.003604,36.324324,3.691892,1,7.0,False
4,2018,1,8,7,16,Wheel,Grosjean,Haas F1 Team,6,24,...,NaN,NaN,24.077477,30.915315,997.003604,36.324324,3.691892,1,7.0,False


In [156]:
trke.columns

Index(['season', 'round', 'drivers_num', 'position_quali', 'position_race',
       'status', 'driver', 'constructor', 'grid', 'laps', 'total_time_ms',
       'points', 'driverId', '_merge', 'race_name', 'circuit', ' lap_length',
       ' number_of_laps', ' number_of_corners', ' pit_lane_time_loss',
       ' average_overtakes_per_circuit', ' average_pit_stops_per_race',
       'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindSpeed', 'rain_bin',
       'wind_direction', 'is_sprint_weekend'],
      dtype='object')

In [157]:
df = pd.DataFrame()
for i in range(2018, 2024):
    last_season = trke[trke['season'] == i][['season', 'driverId', 'race_name', 'total_time_ms']].copy()
    last_season['season'] += 1 
    last_season = last_season.rename(columns={'total_time_ms': 'last_year_time'})
    df = pd.concat([df, last_season])

trke = trke.merge(df, how='left', on=['season', 'driverId', 'race_name'])


In [158]:
trke.columns

Index(['season', 'round', 'drivers_num', 'position_quali', 'position_race',
       'status', 'driver', 'constructor', 'grid', 'laps', 'total_time_ms',
       'points', 'driverId', '_merge', 'race_name', 'circuit', ' lap_length',
       ' number_of_laps', ' number_of_corners', ' pit_lane_time_loss',
       ' average_overtakes_per_circuit', ' average_pit_stops_per_race',
       'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindSpeed', 'rain_bin',
       'wind_direction', 'is_sprint_weekend', 'last_year_time'],
      dtype='object')

In [159]:
len(trke)

2979

In [160]:
len(df)

2500

In [161]:
trke.columns

Index(['season', 'round', 'drivers_num', 'position_quali', 'position_race',
       'status', 'driver', 'constructor', 'grid', 'laps', 'total_time_ms',
       'points', 'driverId', '_merge', 'race_name', 'circuit', ' lap_length',
       ' number_of_laps', ' number_of_corners', ' pit_lane_time_loss',
       ' average_overtakes_per_circuit', ' average_pit_stops_per_race',
       'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindSpeed', 'rain_bin',
       'wind_direction', 'is_sprint_weekend', 'last_year_time'],
      dtype='object')

In [162]:
trke['avg_position_last5'] = np.nan
trke['num_dnfs_last5'] = np.nan
trke['avg_gained_lost_last5'] = np.nan

In [163]:
# racunanje proseka pozicija, broja odutajanja i broja stecenih/izgubljenih pozicija u posednjih 5 trka
for broj in vozaci['permanentNumber']:
    jedan_vozac = trke[trke['drivers_num'] == broj]
    
    jedan_vozac = jedan_vozac.drop(['total_time_ms',
                    'race_name', 'circuit', ' lap_length',
                    ' number_of_laps', ' number_of_corners', ' pit_lane_time_loss',
                    ' average_overtakes_per_circuit', ' average_pit_stops_per_race',
                    'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindSpeed', 'rain_bin',
                    'wind_direction', 'is_sprint_weekend', 'constructor'], axis=1)
    
    jedan_vozac = jedan_vozac.sort_values(by=['season', 'round'])
    for i in range(5, len(jedan_vozac)):
        last5 = jedan_vozac.iloc[i-5:i]
        current_race = jedan_vozac.iloc[i]


        last5 = last5.copy()
        last5["position_race"] = pd.to_numeric(last5["position_race"], errors="coerce")
        last5["position_quali"] = pd.to_numeric(last5["position_quali"], errors="coerce")



        avg_position = last5["position_race"].astype(float).mean()
        #ako nije finished ili ne sadrzi lap onda, je driver dnf
        num_dnfs = (~last5["status"].str.contains("Finished|Lap")).sum()
        gained_lost = (last5["grid"] - last5["position_race"]).astype(float)
        avg_gained_lost = gained_lost.mean()  


        idx = jedan_vozac.index[i]
        trke.loc[idx, 'avg_position_last5'] = avg_position
        trke.loc[idx, 'num_dnfs_last5'] = num_dnfs
        trke.loc[idx, 'avg_gained_lost_last5'] = avg_gained_lost

In [164]:
#jer trenutno sadrze NaN vrednosti
trke = trke.drop([' pit_lane_time_loss',
       ' average_overtakes_per_circuit', ' average_pit_stops_per_race'], axis=1)

In [165]:
trke = trke.sort_values(by=['season', 'round', 'position_race'])

In [166]:
trke.tail()

,season,round,drivers_num,position_quali,position_race,status,driver,constructor,grid,laps,...,Pressure,TrackTemp,WindSpeed,rain_bin,wind_direction,is_sprint_weekend,last_year_time,avg_position_last5,num_dnfs_last5,avg_gained_lost_last5
2965,2024,24,20,15,16,+1 Lap,Magnussen,Haas F1 Team,14,57,...,1017.426351,31.805405,1.900676,0,3.0,False,NaN,11.6,1.0,-1.4
2970,2024,24,30,12,17,Engine,Lawson,RB F1 Team,12,55,...,1017.426351,31.805405,1.900676,0,3.0,False,NaN,12.8,0.0,0.6
2977,2024,24,77,9,18,Collision damage,Bottas,Sauber,9,30,...,1017.426351,31.805405,1.900676,0,3.0,False,NaN,14.6,0.0,0.2
2972,2024,24,43,19,19,Engine,Colapinto,Williams,20,26,...,1017.426351,31.805405,1.900676,0,3.0,False,NaN,14.4,2.0,-1.2
2961,2024,24,11,10,20,Collision,Pérez,Red Bull,10,0,...,1017.426351,31.805405,1.900676,0,3.0,False,5244077.0,12.4,1.0,0.2


In [167]:
trke.columns

Index(['season', 'round', 'drivers_num', 'position_quali', 'position_race',
       'status', 'driver', 'constructor', 'grid', 'laps', 'total_time_ms',
       'points', 'driverId', '_merge', 'race_name', 'circuit', ' lap_length',
       ' number_of_laps', ' number_of_corners', 'AirTemp', 'Humidity',
       'Pressure', 'TrackTemp', 'WindSpeed', 'rain_bin', 'wind_direction',
       'is_sprint_weekend', 'last_year_time', 'avg_position_last5',
       'num_dnfs_last5', 'avg_gained_lost_last5'],
      dtype='object')

In [ ]:
trke.to_csv('tables//trke_sa_driverId.csv', index=False)